## Assignment no 4

# BASIC STATISTICS-2


Problem Statement : Hospital Patient Data Analysis

Context:

A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

Tasks:

1.	Load the patient dataset and show summary with info().
2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].
3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].
4.	Use groupby to find total bill amount per department.
5.	Remove duplicate patient records based on PatientID.
6.	Fill missing BillAmount values with the mean bill amount.
7.	Merge the billing dataset with patient dataset on PatientID.
8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).
9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

Expected Outcome:

•	Final cleaned dataset with accurate billing info.
•	All missing values handled, merged dataset across PatientID.
•	Ability to perform further analytics on department-wise revenue or doctor performance.



Task 1 — Load & info()

In [1]:
import pandas as pd
import numpy as np

patient = pd.read_csv("Patient_Data.csv")
billing = pd.read_csv("Billing_Data.csv")

patient.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


Task 2 — Select billing‑relevant columns

In [2]:
bill_cols = ['PatientID', 'Department', 'Doctor', 'BillAmount']
patient_bill = patient[bill_cols].copy()
print(patient_bill)

   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN
5        101   Cardiology  Dr. Smith      5000.0


Task 3 — Drop administrative columns

In [3]:
admin_cols = ['ReceptionistID', 'CheckInTime']
patient_clean = patient.drop(columns=admin_cols)
print(patient_clean.columns.tolist())

['PatientID', 'Name', 'Department', 'Doctor', 'BillAmount']


Task 4 — Groupby: total bill per department

In [4]:
dept_total = (patient_clean
              .groupby('Department')['BillAmount']
              .agg(total='sum', count='count', mean='mean'))
print(dept_total)

               total  count    mean
Department                         
Cardiology   16200.0      3  5400.0
Dermatology      0.0      0     NaN
Neurology        0.0      0     NaN
Orthopedics   7500.0      1  7500.0


Task 5 — Remove duplicates on PatientID

In [5]:
patient_dedup = patient_clean.drop_duplicates(subset='PatientID', keep='first')
print(patient_dedup)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


Task 6 — Fill missing BillAmount with the mean

In [6]:
mean_bill = patient_dedup['BillAmount'].mean()   # (5000 + 7500 + 6200) / 3
print(f"Mean bill (post-dedupe): ₹{mean_bill:,.2f}")

patient_filled = patient_dedup.copy()
patient_filled['BillAmount'] = patient_filled['BillAmount'].fillna(mean_bill)
print(patient_filled)

Mean bill (post-dedupe): ₹6,233.33
   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


Task 7 — Merge patient and billing on PatientID

In [7]:
# Dedupe billing first, then keep only billing-specific columns
billing_clean = (billing
                 .drop_duplicates(subset='PatientID', keep='first')
                 [['PatientID', 'BillAmount']]
                 .rename(columns={'BillAmount': 'BillAmount_Billing'}))

merged = pd.merge(patient_filled, billing_clean,
                  on='PatientID', how='left')
print(merged)

   PatientID     Name   Department     Doctor   BillAmount  BillAmount_Billing
0        101    Alice   Cardiology  Dr. Smith  5000.000000              5000.0
1        102      Bob    Neurology   Dr. John  6233.333333                 NaN
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              7500.0
3        104    David   Cardiology  Dr. Smith  6200.000000              6200.0
4        105      Eva  Dermatology   Dr. Rose  6233.333333                 NaN


Task 8 — Row-wise concatenation (new patients this week)

In [8]:
new_patients = pd.DataFrame({
    'PatientID':  [106, 107],
    'Name':       ['Frank', 'Grace'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor':     ['Dr. Smith', 'Dr. John'],
    'BillAmount': [4800.0, 5400.0],
    'BillAmount_Billing': [4800.0, 5400.0]
})

combined = pd.concat([merged, new_patients], ignore_index=True)
print(combined)

   PatientID     Name   Department     Doctor   BillAmount  BillAmount_Billing
0        101    Alice   Cardiology  Dr. Smith  5000.000000              5000.0
1        102      Bob    Neurology   Dr. John  6233.333333                 NaN
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              7500.0
3        104    David   Cardiology  Dr. Smith  6200.000000              6200.0
4        105      Eva  Dermatology   Dr. Rose  6233.333333                 NaN
5        106    Frank   Cardiology  Dr. Smith  4800.000000              4800.0
6        107    Grace    Neurology   Dr. John  5400.000000              5400.0


Task 9 — Column-wise concatenation (billing categories)

In [9]:
billing_categories = pd.DataFrame({
    'InsuranceCovered': [2000, 1500, 3000, 2500, 1000, 1800, 2200],
    'FinalAmount':      [3000, 4733.33, 4500, 3700, 5233.33, 3000, 3200]
})

final_df = pd.concat([combined, billing_categories], axis=1)
print(final_df)
print("\nShape:", final_df.shape)   # (7, 8)

   PatientID     Name   Department     Doctor   BillAmount  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000   
1        102      Bob    Neurology   Dr. John  6233.333333   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000   
3        104    David   Cardiology  Dr. Smith  6200.000000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333   
5        106    Frank   Cardiology  Dr. Smith  4800.000000   
6        107    Grace    Neurology   Dr. John  5400.000000   

   BillAmount_Billing  InsuranceCovered  FinalAmount  
0              5000.0              2000      3000.00  
1                 NaN              1500      4733.33  
2              7500.0              3000      4500.00  
3              6200.0              2500      3700.00  
4                 NaN              1000      5233.33  
5              4800.0              1800      3000.00  
6              5400.0              2200      3200.00  

Shape: (7, 8)


Final Cleaned Dataset

In [10]:
print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PatientID           7 non-null      int64  
 1   Name                7 non-null      object 
 2   Department          7 non-null      object 
 3   Doctor              7 non-null      object 
 4   BillAmount          7 non-null      float64
 5   BillAmount_Billing  5 non-null      float64
 6   InsuranceCovered    7 non-null      int64  
 7   FinalAmount         7 non-null      float64
dtypes: float64(3), int64(2), object(3)
memory usage: 580.0+ bytes
None


Post‑Cleanup Analytics
Department-wise revenue (clean)

In [11]:
(final_df.groupby('Department')
    .agg(
        total_billed   = ('BillAmount', 'sum'),
        total_insured  = ('InsuranceCovered', 'sum'),
        total_payable  = ('FinalAmount', 'sum'),
        patients       = ('PatientID', 'nunique'),
    )
    .sort_values('total_payable', ascending=False))

,total_billed,total_insured,total_payable,patients
Department,,,,
Cardiology,16000.000000,6300,9700.00,3
Neurology,11633.333333,3700,7933.33,2
Dermatology,6233.333333,1000,5233.33,1
Orthopedics,7500.000000,3000,4500.00,1


Doctor performance

In [12]:
(final_df.groupby('Doctor')
    .agg(patients=('PatientID','count'),
         avg_bill=('BillAmount','mean'),
         revenue=('FinalAmount','sum'))
    .round(2))

,patients,avg_bill,revenue
Doctor,,,
Dr. John,2,5816.67,7933.33
Dr. Lee,1,7500.00,4500.00
Dr. Rose,1,6233.33,5233.33
Dr. Smith,3,5333.33,9700.00


10. Save final cleaned datasets

In [ ]:

patients.to_csv("cleaned_patient_data.csv", index=False)
billing.to_csv("cleaned_billing_data.csv", index=False)
merged_data.to_csv("merged_patient_billing_data.csv", index=False)
all_patients.to_csv("final_hospital_billing_data.csv", index=False)

print("\nAll tasks completed successfully!")


All tasks completed successfully!
